In [13]:
from __future__ import annotations
import hashlib
import json
import time
from copy import deepcopy
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

Block = dict[str, Any]
Chain = list[Block]

DIFFICULTY = 3  


def _sha(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def _header_hash(index, timestamp, txs, previous_hash, nonce) -> str:
    payload = json.dumps(
        {
            "index": index,
            "timestamp": timestamp,
            "transactions": txs,
            "previous_hash": previous_hash,
            "nonce": nonce,
        },
        sort_keys=True,
    )
    return _sha(payload)


def mine_block(
    index, txs, previous_hash, difficulty: int = DIFFICULTY
) -> Block:
    """Proof-of-work search until the hash has `difficulty` leading hex zeros."""
    timestamp = time.time()
    nonce = 0
    while True:
        h = _header_hash(index, timestamp, txs, previous_hash, nonce)
        if h.startswith("0" * difficulty):
            return {
                "index": index,
                "timestamp": timestamp,
                "transactions": txs,
                "previous_hash": previous_hash,
                "nonce": nonce,
                "hash": h,
                "difficulty": difficulty,
            }
        nonce += 1


def make_genesis() -> Block:
    return mine_block(0, [{"note": "genesis"}], "0" * 64)


genesis = make_genesis()
print("Genesis hash:", genesis["hash"][:16], "...")
print("Difficulty:", genesis["difficulty"], "| nonce:", genesis["nonce"])

Genesis hash: 0009693e594e4643 ...
Difficulty: 3 | nonce: 449


In [14]:
def verify_chain(chain: Chain, difficulty: int = DIFFICULTY) -> bool:
    """Return True iff hashes, parent links, and PoW difficulty checks pass."""
    if not chain:
        return False
    for i, block in enumerate(chain):
        expected = _header_hash(
            block["index"],
            block["timestamp"],
            block["transactions"],
            block["previous_hash"],
            block["nonce"],
        )
        if block.get("hash") != expected:
            return False
        diff = int(block.get("difficulty", difficulty))
        if not str(block["hash"]).startswith("0" * diff):
            return False
        if i == 0:
            continue
        if block["previous_hash"] != chain[i - 1]["hash"]:
            return False
        if block["index"] != i:
            return False
    return True


print("Genesis alone verifies:", verify_chain([genesis]))

# Junk dicts must not pass
junk = [deepcopy(genesis)] + [
    {"index": i, "hash": "ff" * 32, "previous_hash": "x", "nonce": 0,
     "timestamp": 0, "transactions": [], "difficulty": DIFFICULTY}
    for i in range(1, 6)
]
print("Long junk list verifies:", verify_chain(junk), "(must be False)")

Genesis alone verifies: True
Long junk list verifies: False (must be False)


In [15]:
def chain_work(chain: Chain) -> int:
    """Heaviest-work score: sum of per-block difficulties.

    Equals length ranking when every block shares the same difficulty.
    """
    total = 0
    for block in chain:
        total += int(block.get("difficulty", DIFFICULTY))
    return total


@dataclass
class Node:
    """A simulated peer holding its own view of the chain."""

    name: str
    chain: Chain = field(default_factory=list)

    def tip(self) -> Block:
        if not self.chain:
            raise ValueError(f"node {self.name} has an empty chain")
        return self.chain[-1]

    def height(self) -> int:
        return len(self.chain) - 1 if self.chain else -1

    def summary(self) -> str:
        tip_short = self.tip()["hash"][:10]
        return (
            f"{self.name}: height={self.height()} tip={tip_short}... "
            f"valid={verify_chain(self.chain)} work={chain_work(self.chain)}"
        )


A = Node("A", [deepcopy(genesis)])
B = Node("B", [deepcopy(genesis)])
C = Node("C", [deepcopy(genesis)])

for n in (A, B, C):
    print(n.summary())
print("Shared tip?", A.tip()["hash"] == B.tip()["hash"] == C.tip()["hash"])

A: height=0 tip=0009693e59... valid=True work=3
B: height=0 tip=0009693e59... valid=True work=3
C: height=0 tip=0009693e59... valid=True work=3
Shared tip? True


In [ ]:
#Question 2

In [16]:
tx_alice = [
    {
        "id": "pay-alice",
        "sender": "bank",
        "recipient": "Alice",
        "amount": 10000,
        "currency": "ZAR",
    }
]
tx_bob = [
    {
        "id": "pay-bob",
        "sender": "bank",
        "recipient": "Bob",
        "amount": 10000,
        "currency": "ZAR",
    }
]

block_A1 = mine_block(1, tx_alice, A.tip()["hash"])
block_B1 = mine_block(1, tx_bob, B.tip()["hash"])

A.chain.append(block_A1)
B.chain.append(block_B1)
# C still at genesis only

print("A and B tips differ?", A.tip()["hash"] != B.tip()["hash"])
print("Both extensions valid?",
      verify_chain(A.chain), "|", verify_chain(B.chain))
print("Same parent (genesis)?",
      A.chain[1]["previous_hash"] == B.chain[1]["previous_hash"] == genesis["hash"])
for n in (A, B, C):
    print(" ", n.summary())

print("\nBoard sketch:")
print("  A: G -> 1_A (pay-alice)")
print("  B: G -> 1_B (pay-bob)")
print("  C: G")

A and B tips differ? True
Both extensions valid? True | True
Same parent (genesis)? True
  A: height=1 tip=00065c3749... valid=True work=6
  B: height=1 tip=00091fec5d... valid=True work=6
  C: height=0 tip=0009693e59... valid=True work=3

Board sketch:
  A: G -> 1_A (pay-alice)
  B: G -> 1_B (pay-bob)
  C: G


In [17]:
def chain_length(chain: Chain) -> int:
    """Longest-chain score (for comparison demos only)."""
    return len(chain)


print(f"A: length={chain_length(A.chain)} work={chain_work(A.chain)}")
print(f"B: length={chain_length(B.chain)} work={chain_work(B.chain)}")
print(f"C: length={chain_length(C.chain)} work={chain_work(C.chain)}")
print("Equal scores at the fork tip -> need a tie policy.")

print("\nWhen difficulty varies (illustrative scores, not mined here):")
print("  long easy:  length=5, work=5  (five difficulty-1 blocks)")
print("  short hard: length=3, work=9  (three difficulty-3 blocks)")
print("  longest picks easy; heaviest picks hard")
print("Under uniform classroom DIFFICULTY they coincide - still name both in the A7 report.")

A: length=2 work=6
B: length=2 work=6
C: length=1 work=3
Equal scores at the fork tip -> need a tie policy.

When difficulty varies (illustrative scores, not mined here):
  long easy:  length=5, work=5  (five difficulty-1 blocks)
  short hard: length=3, work=9  (three difficulty-3 blocks)
  longest picks easy; heaviest picks hard
Under uniform classroom DIFFICULTY they coincide - still name both in the A7 report.


In [18]:
def tie_break(chains: list[Chain]) -> Chain:
    """Lexicographically smaller tip hash (documented classroom policy)."""
    return min(chains, key=lambda c: c[-1]["hash"])


def resolve_fork(
    chains: list[Chain], verifier: Callable[[Chain], bool] | None = None
) -> Chain:
    """Among valid chains, return a deep copy of the winning chain."""
    verify = verifier or verify_chain
    valid = [c for c in chains if verify(c)]
    if not valid:
        raise ValueError("no valid chain")
    best = max(chain_work(c) for c in valid)
    tied = [c for c in valid if chain_work(c) == best]
    chosen = tied[0] if len(tied) == 1 else tie_break(tied)
    return deepcopy(chosen)


# Equal work at height 1 
winner_tied = resolve_fork([A.chain, B.chain, C.chain])
w1 = resolve_fork([A.chain, B.chain])
w2 = resolve_fork([B.chain, A.chain])
print("Tie at equal work - tip chosen:", winner_tied[-1]["hash"][:16], "...")
print("Order-independent?", w1[-1]["hash"] == w2[-1]["hash"])
print("Contains pay-alice?",
      any(tx.get("id") == "pay-alice" for tx in winner_tied[-1].get("transactions", [])))
print("Contains pay-bob?",
      any(tx.get("id") == "pay-bob" for tx in winner_tied[-1].get("transactions", [])))

# Invalid chain must never win even if longer
invalid_long = deepcopy(A.chain)
fake = dict(invalid_long[-1])
fake["hash"] = "00" + "f" * 62  # looks PoW-ish but hash does not recompute
invalid_long[-1] = fake
invalid_long.append(
    {"index": 2, "timestamp": 0, "transactions": [], "previous_hash": fake["hash"],
     "nonce": 0, "hash": "00" + "e" * 62, "difficulty": DIFFICULTY}
)
resolved = resolve_fork([invalid_long, B.chain])
print("Invalid longer chain rejected; winner height:", resolved[-1]["index"])
print("Winner valid?", verify_chain(resolved))

Tie at equal work - tip chosen: 00065c37490f2ec5 ...
Order-independent? True
Contains pay-alice? True
Contains pay-bob? False
Invalid longer chain rejected; winner height: 1
Winner valid? True


In [ ]:
#Question 3

In [19]:
def sync_nodes(nodes: list[Node]) -> Chain:
    """Resolve all node chains and set every node to the winner (deep copy)."""
    winner = resolve_fork([n.chain for n in nodes])
    for node in nodes:
        node.chain = deepcopy(winner)
    return winner


# Restore divergent state for the demo
A.chain = [deepcopy(genesis), deepcopy(block_A1)]
B.chain = [deepcopy(genesis), deepcopy(block_B1)]
C.chain = [deepcopy(genesis)]

block_A2 = mine_block(2, [{"note": "extend-A"}], A.tip()["hash"])
A.chain.append(block_A2)

print("Before sync:")
for n in (A, B, C):
    print(" ", n.summary())

heights_before = {"A": A.height(), "B": B.height(), "C": C.height()}
canonical = sync_nodes([A, B, C])

print("After sync (heaviest / longest under uniform D):")
for n in (A, B, C):
    print(" ", n.summary())
print("Heights before:", heights_before)
print("Shared tip after:", A.tip()["hash"][:16], "...")
print("Canonical payment id:", A.chain[1]["transactions"][0].get("id"))

Before sync:
  A: height=2 tip=000980015e... valid=True work=9
  B: height=1 tip=00091fec5d... valid=True work=6
  C: height=0 tip=0009693e59... valid=True work=3
After sync (heaviest / longest under uniform D):
  A: height=2 tip=000980015e... valid=True work=9
  B: height=2 tip=000980015e... valid=True work=9
  C: height=2 tip=000980015e... valid=True work=9
Heights before: {'A': 2, 'B': 1, 'C': 0}
Shared tip after: 000980015e228cb8 ...
Canonical payment id: pay-alice


In [20]:
def block_ids_on_chain(chain: Chain) -> set[str]:
    return {b["hash"] for b in chain}


canonical_hashes = block_ids_on_chain(canonical)
orphan_candidate = block_B1  

is_orphan = orphan_candidate["hash"] not in canonical_hashes
orphan_still_valid_alone = verify_chain([deepcopy(genesis), deepcopy(orphan_candidate)])

print("Losing tip hash:", orphan_candidate["hash"][:16], "...")
print("On canonical chain?", not is_orphan)
print("Still a valid extension of genesis by itself?", orphan_still_valid_alone)
print("Tx trapped on orphan:", orphan_candidate["transactions"][0]["id"])
print("=> pay-bob is absent from the shared ledger until re-mined on the winning tip.")

Losing tip hash: 00091fec5d628311 ...
On canonical chain? False
Still a valid extension of genesis by itself? True
Tx trapped on orphan: pay-bob
=> pay-bob is absent from the shared ledger until re-mined on the winning tip.


In [21]:
# Rebuild a fresh C and a preserved copy of A's winning chain
chain_B_short = [deepcopy(genesis), deepcopy(block_B1)]
chain_A_long = deepcopy(canonical)  # G -> 1_A -> 2_A

C.chain = [deepcopy(genesis)]
print("C start:", C.summary())

# First rumour: B's tip arrives
C.chain = resolve_fork([C.chain, chain_B_short])
print("C after hearing B:", C.summary())
print("  local payment id:", C.chain[1]["transactions"][0].get("id"))

# Later: A's heavier chain arrives -> reorg
before_tip = C.tip()["hash"]
C.chain = resolve_fork([C.chain, chain_A_long])
after_tip = C.tip()["hash"]
print("C after hearing A (heavier):", C.summary())
print("  reorg occurred?", before_tip != after_tip)
print("  local payment id now:", C.chain[1]["transactions"][0].get("id"))

C start: C: height=0 tip=0009693e59... valid=True work=3
C after hearing B: C: height=1 tip=00091fec5d... valid=True work=6
  local payment id: pay-bob
C after hearing A (heavier): C: height=2 tip=000980015e... valid=True work=9
  reorg occurred? True
  local payment id now: pay-alice


In [22]:
def find_tx_height(chain: Chain, tx_id: str) -> int | None:
    """Return the block index containing tx_id, or None."""
    for block in chain:
        for tx in block.get("transactions", []):
            if tx.get("id") == tx_id:
                return int(block["index"])
    return None


def confirmations(chain: Chain, tx_id: str) -> int:
    """Confirmation depth k for a transaction id.

    If included at height h and tip height is H, k = H - h + 1.
    Return 0 if the transaction is absent.
    """
    h = find_tx_height(chain, tx_id)
    if h is None or not chain:
        return 0
    tip_h = int(chain[-1]["index"])
    return tip_h - h + 1


# Arithmetic drill: h=4, H=9 -> k=6
assert (9 - 4 + 1) == 6

pay_id = "pay-alice"
print(f"{pay_id!r} height:", find_tx_height(A.chain, pay_id))
print(f"{pay_id!r} confirmations at current tip:", confirmations(A.chain, pay_id))
print(f"{'pay-bob'!r} confirmations (orphaned):", confirmations(A.chain, "pay-bob"))

# Pad the tip to deepen confirmations
for i in range(A.height() + 1, 6):
    nxt = mine_block(i, [{"note": f"padding-{i}"}], A.tip()["hash"])
    A.chain.append(nxt)

print("After padding to height", A.height(),
      "-> confirmations:", confirmations(A.chain, pay_id))
print("Missing tx ->", confirmations(A.chain, "missing"))

'pay-alice' height: 1
'pay-alice' confirmations at current tip: 2
'pay-bob' confirmations (orphaned): 0
After padding to height 5 -> confirmations: 5
Missing tx -> 0


In [23]:
REQUIRED_K_COFFEE = 1   # fast UX, high reorg exposure
REQUIRED_K_OTC = 6      # slower UX, lower reorg risk
REQUIRED_K_CLASS_POLICY = 3  # example justified policy for the toy chain

k_now = confirmations(A.chain, "pay-alice")


def settled_for_policy(chain: Chain, tx_id: str, required_k: int) -> bool:
    return confirmations(chain, tx_id) >= required_k


print(f"pay-alice confirmations now: {k_now}")
print(f"  coffee-shop (k>={REQUIRED_K_COFFEE}):",
      settled_for_policy(A.chain, "pay-alice", REQUIRED_K_COFFEE))
print(f"  class policy (k>={REQUIRED_K_CLASS_POLICY}):",
      settled_for_policy(A.chain, "pay-alice", REQUIRED_K_CLASS_POLICY))
print(f"  OTC desk (k>={REQUIRED_K_OTC}):",
      settled_for_policy(A.chain, "pay-alice", REQUIRED_K_OTC))

print("\nQualitative check for the report:")
print("  Ticket size: R10,000")
print("  Toy difficulty:", DIFFICULTY, "(not production security)")
print("  Chosen classroom k:", REQUIRED_K_CLASS_POLICY)
print("  Justify: reorg must rewrite k blocks of work; larger ticket => usually larger k.")

pay-alice confirmations now: 5
  coffee-shop (k>=1): True
  class policy (k>=3): True
  OTC desk (k>=6): False

Qualitative check for the report:
  Ticket size: R10,000
  Toy difficulty: 3 (not production security)
  Chosen classroom k: 3
  Justify: reorg must rewrite k blocks of work; larger ticket => usually larger k.


In [24]:
def demo_fork_scenario() -> dict[str, Any]:
    """Build genesis, fork A/B, extend A, sync, return a summary dict."""
    g = mine_block(0, [{"note": "genesis"}], "0" * 64)
    a = Node("A", [deepcopy(g)])
    b = Node("B", [deepcopy(g)])
    c = Node("C", [deepcopy(g)])

    a.chain.append(
        mine_block(1, [{"id": "pay-alice", "amount": 10000}], a.tip()["hash"])
    )
    b.chain.append(
        mine_block(1, [{"id": "pay-bob", "amount": 10000}], b.tip()["hash"])
    )
    a.chain.append(mine_block(2, [{"note": "extend-A"}], a.tip()["hash"]))

    before = {"A": a.height(), "B": b.height(), "C": c.height()}
    sync_nodes([a, b, c])
    return {
        "heights_before": before,
        "tip_after": a.tip()["hash"],
        "canonical_tx": a.chain[1]["transactions"][0]["id"],
        "confirmations_alice": confirmations(a.chain, "pay-alice"),
        "confirmations_bob": confirmations(a.chain, "pay-bob"),
    }


summary = demo_fork_scenario()
print(json.dumps(summary, indent=2))
assert summary["canonical_tx"] == "pay-alice"
assert summary["confirmations_bob"] == 0
assert summary["confirmations_alice"] >= 1
print("demo_fork_scenario checks passed.")

{
  "heights_before": {
    "A": 2,
    "B": 1,
    "C": 0
  },
  "tip_after": "000d81ebe2084d8684d306bba40c9da78281484549bb950fdaa5f3f0640f8dce",
  "canonical_tx": "pay-alice",
  "confirmations_alice": 2,
  "confirmations_bob": 0
}
demo_fork_scenario checks passed.
